In [253]:
import gzip
from tqdm import tqdm
import matplotlib.pyplot as plt
from collections import defaultdict
import numpy as np

In [254]:
def readGz(path):
    for l in gzip.open(path, 'rt'):
        yield eval(l)

def readCSV(path):
    f = gzip.open(path, 'rt')
    f.readline()
    for l in f:
        yield l.strip().split(',')

In [255]:
allRatings = [(u, b, int(r)) for u, b, r in readCSV("train_Interactions.csv.gz")]

ratingsTrain = allRatings[:190000]
ratingsValid = allRatings[190000:]
ratingsPerUser = defaultdict(list)
ratingsPerItem = defaultdict(list)

for u, b, r in ratingsTrain:
    ratingsPerUser[u].append((b, r))
    ratingsPerItem[b].append((u, r))

trainRatings = [r[2] for r in ratingsTrain]
globalAverage = sum(trainRatings) / len(trainRatings)

In [256]:
def alphaUpdate(ratingsTrain, alpha, betaU, betaI, lamb):
    return sum(r[2] - betaU[r[0]] - betaI[r[1]] for r in ratingsTrain) / len(ratingsTrain)

def betaUUpdate(ratingsPerUser, alpha, betaU, betaI, lamb):
    return {
        u: sum(r - alpha - betaI[i] for i, r in ratingsPerUser[u]) / (lamb + len(ratingsPerUser[u]))
        for u in ratingsPerUser
    }

def betaIUpdate(ratingsPerItem, alpha, betaU, betaI, lamb):
    return {
        i: sum(r - alpha - betaU[u] for u, r in ratingsPerItem[i]) / (lamb + len(ratingsPerItem[i]))
        for i in ratingsPerItem
    }

In [257]:
# Train model with best parameters (lambda=5.0, C=100)
lamb = 4.5
steps = 50

betaU = {u: 0 for u in ratingsPerUser}
betaI = {b: 0 for b in ratingsPerItem}
alpha = globalAverage

# Train model
for _ in tqdm(range(steps), desc="Training rating model"):
    alpha = alphaUpdate(ratingsTrain, alpha, betaU, betaI, lamb)
    betaU = betaUUpdate(ratingsPerUser, alpha, betaU, betaI, lamb)
    betaI = betaIUpdate(ratingsPerItem, alpha, betaU, betaI, lamb)

Training rating model: 100%|██████████| 50/50 [00:18<00:00,  2.65it/s]


In [258]:
def predictRating(u, b, **kwargs):
    return kwargs["alpha"] + kwargs["betaU"].get(u, 0) + kwargs["betaI"].get(b, 0)

def modpredictRating(u, b, **kwargs):
    baseline = predictRating(u, b, **kwargs)
    if b in kwargs["ratingsPerItem"] and len(kwargs["ratingsPerItem"][b]) > 0:
        item_avg = sum(r for _, r in kwargs["ratingsPerItem"][b]) / len(kwargs["ratingsPerItem"][b])
        if item_avg > baseline:
            return baseline - 0.115*(item_avg - baseline)
    return baseline

def MSE(ratings, predict, **kwargs):
    return sum((r - predict(u, i, **kwargs)) ** 2 for u, i, r in ratings) / len(ratings)

In [259]:
kwargs = {
    "alpha": alpha,
    "betaU": betaU,
    "betaI": betaI,
    "ratingsPerItem": ratingsPerItem
}

predictions = open("predictions_Rating.csv", "w")
for l in open("pairs_Rating.csv"):
    if l.startswith("userID"):
        predictions.write(l)
        continue
    u, b = l.strip().split(",")
    pred = max(0.0, min(5.0, modpredictRating(u, b, **kwargs)))
    predictions.write(f"{u},{b},{pred}\n")
predictions.close()